# Langgraph 실습

## State 기획 필요

## Node
1. START - 사용자의 아픈 증상을 메세지 입력
2. 메세지를 기반으로 증상 추출 (LLM)
3. 증상별로 도움되는 영양소를 뽑아줌 **(LLM with Structured Output)** - Agent 아님. `llm.invoke`
    - `symtom: str` -> 증상
    - `nutrients: list[str]` -> 추천하는 영양소 목록
    - `reason: str` ->  각 영양소를 추천한 종합 소견
4. 증상 + 영양소 목록으로 최종 안내문 생성 (LLM)
5. END

In [39]:
import os
from dotenv import load_dotenv
from typing_extensions import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field #llm에게 OutputParser 를 입력해 주기 위한 import

load_dotenv()

True

In [41]:
#State정의
class DiagDisease(TypedDict):
    message: str
    symtom: str 
    nutrients: List[str]  
    reason: str 

class ExtractSymtom(BaseModel): # **sturctured Output의 반환 값은 pydantic instance(객체) 형태로 반환됨 dict아님!**
    symtom: str = Field(description="사용자 메시지에서 추출한 증상")

class ExtractNutrients(BaseModel):
    nutrients: List[str] = Field(description="증상으로 부터 추천하는 영양소")  

In [43]:
#증상 체크
def check_symtom(state : DiagDisease):
  
  llm = init_chat_model('openai:gpt-4.1-mini')
  structured_llm = llm.with_structured_output(ExtractSymtom) ## 반환값 -> {'symtom':'Ai가 판단한 증상'}

  result = structured_llm.invoke([
    {'role':'system', 'content':'너는 사용자 메세지를 읽고 증상을 분류해주는 AI야'},
    {'role':'user', 'content': state['message']},
  ])
  return {'symtom': result.symtom } # result는 pydantic 객체 형태이기 때문에 
                                    # 1.result.symtom -> 객체의 변수값에 직접 접근(반환값 = 해당 변수의 값) or
                                    # 2.result.model_dump() -> pydantic 내장 함수로 반환 해야함.(반환값 = dict)
#영양소 추천
def recommend_nutrients(state : DiagDisease):

  llm = init_chat_model('openai:gpt-4.1-mini')
  structured_llm = llm.with_structured_output(ExtractNutrients)

  result = structured_llm.invoke([
    {'role':'system', 'content':'너는 입력된 증상을 바탕으로 그 증상에 좋은 영양소를 추천해주는 AI야'},
    {'role':'user', 'content': state['symtom']},
  ]) 

  return{'nutrients': result.nutrients}

#답변 생성
def make_reason(state : DiagDisease):
  reason = f'현재 발생한 {state['symtom']}에 추천하는 영양소는 {state['nutrients']}입니다.'
  return{'reason': reason}


In [44]:
builder = StateGraph(DiagDisease)

builder.add_node('check_symtom', check_symtom)
builder.add_node('recommend_nutrients', recommend_nutrients)
builder.add_node('make_reason', make_reason)

builder.add_edge(START, 'check_symtom')
builder.add_edge('check_symtom', 'recommend_nutrients')
builder.add_edge('recommend_nutrients', 'make_reason')
builder.add_edge('make_reason', END)

graph = builder.compile()

In [45]:
graph.invoke({
  'message' : '나 구토와 멀미가나'
})


{'message': '나 구토와 멀미가나',
 'symtom': '구토와 멀미',
 'nutrients': ['생강 추출물', '비타민 B6', '수분 보충용 전해질', '필요시 프로바이오틱스'],
 'reason': "현재 발생한 구토와 멀미에 추천하는 영양소는 ['생강 추출물', '비타민 B6', '수분 보충용 전해질', '필요시 프로바이오틱스']입니다."}